In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower


 Exercise 1: Calculating Required Sample Size
You are planning an A/B test to evaluate the impact of a new email subject line on the open rate. Based on past data, you expect a small effect size of 0.3 (an increase from 20% to 23% in the open rate). You aim for an 80% chance (power = 0.8) of detecting this effect if it exists, with a 5% significance level (α = 0.05).

Calculate the required sample size per group using Python’s statsmodels library.
What sample size is needed for each group to ensure your test is properly powered?

In [13]:
power =NormalIndPower()
n_sample = power.solve_power(power=0.8, alpha =0.05, effect_size =0.3,alternative ='two-sided')
print(f'{n_sample:.0f} is required  per group for test to be powered')

174 is required  per group for test to be powered


 Exercise 2: Understanding the Relationship Between Effect Size and Sample Size
Using the same A/B test setup as in Exercise 1, you want to explore how changing the expected effect size impacts the required sample size.

Calculate the required sample size for the following effect sizes: 0.2, 0.4, and 0.5, keeping the significance level and power the same.
How does the sample size change as the effect size increases? Explain why this happens.


In [19]:
sample_sizes=[ ]
effect_sizes =[0.2,0.4,0.5]
for size in effect_sizes:
    n=power.solve_power(power=0.8, alpha =0.05, effect_size =size,alternative ='two-sided')
    sample_sizes.append(n)
sizes_compared ={'effect sizes':effect_sizes, 'sample sizes per group':sample_sizes}
df=pd.DataFrame(sizes_compared)
df.head()

,effect sizes,sample sizes per group
0,0.2,392.443023
1,0.4,98.110757
2,0.5,62.790884


the lower the effect size, the higher the sample required to able to detect the correct difference between the variables.

Exercise 3: Exploring the Impact of Statistical Power
Imagine you are conducting an A/B test where you expect a small effect size of 0.2. You initially plan for a power of 0.8 but wonder how increasing or decreasing the desired power level impacts the required sample size.

Calculate the required sample size for power levels of 0.7, 0.8, and 0.9, keeping the effect size at 0.2 and significance level at 0.05.
Question: How does the required sample size change with different levels of statistical power? Why is this understanding important when designing A/B tests?

In [21]:
sample_list=[]
powers=[0.7,0.8,0.9]
for p in powers:
    n=power.solve_power(power =p, alpha =0.02, effect_size =0.2)
    sample_list.append(n)
ab_dict ={'power':powers, 'n_sample':sample_list}
df =pd.DataFrame(ab_dict)
df.head()

,power,n_sample
0,0.7,406.338226
1,0.8,501.801391
2,0.9,650.846915


the higher the power of the A/B test, the higher the sample size required.
This knowledge is very useful when a you have set a target of getting a correct results which depends mostly o  the sample_size.

Exercise 4: Implementing Sequential Testing
You are running an A/B test on two versions of a product page to increase the purchase rate. You plan to monitor the results weekly and stop the test early if one version shows a significant improvement.

Define your stopping criteria.
Decide how you would implement sequential testing in this scenario.
At the end of week three, Version B has a p-value of 0.02. What would you do next?




In [23]:
n_checks = 4
alpha = 0.05
adjusted_alpha = alpha / n_checks


# 2. Simulate weekly results
weekly_data = [
    {"week": 1, "p_value": 0.15, "n_users": 500},
    {"week": 2, "p_value": 0.08, "n_users": 1000},
    {"week": 3, "p_value": 0.02, "n_users": 1500},
    {"week": 4, "p_value": 0.001, "n_users": 2000}
]

stopped = False
print(f"{'Week':<6} {'Users':<8} {'p-value':<10} {'Threshold':<12} {'Decision'}")
print("-" * 56)

stopped = False
for check in weekly_data:
    if stopped:
        decision = "(already stopped)"
    elif check["p_value"] < adjusted_alpha:
        decision = "STOP — Significant!"
        stopped = True
    else:
        decision = "Continue testing"

    print(f"{check['week']:<6} {check['n_users']:<8} {check['p_value']:<10.4f} "
          f"{adjusted_alpha:<12.4f} {decision}")

# 3. Week 3 scenario
print()
print("WEEK 3 SCENARIO: p = 0.02")
print("-" * 40)
if 0.02 < adjusted_alpha:
    print(f"  0.02 < {adjusted_alpha:.4f} -> STOP! Significant.")
else:
    print(f"  0.02 > {adjusted_alpha:.4f} -> CONTINUE!")
    print(f"  Even though 0.02 < 0.05, it's NOT < the stricter threshold.")


Week   Users    p-value    Threshold    Decision
--------------------------------------------------------
1      500      0.1500     0.0125       Continue testing
2      1000     0.0800     0.0125       Continue testing
3      1500     0.0200     0.0125       Continue testing
4      2000     0.0010     0.0125       STOP — Significant!

WEEK 3 SCENARIO: p = 0.02
----------------------------------------
  0.02 > 0.0125 -> CONTINUE!
  Even though 0.02 < 0.05, it's NOT < the stricter threshold.


 Exercise 5: Applying Bayesian A/B Testing
You’re testing a new feature in your app, and you want to use a Bayesian approach. Initially, you believe the new feature has a 50% chance of improving user engagement. After collecting data, your analysis suggests a 65% probability that the new feature is better.

Describe how you would set up your prior belief.
After collecting data, how does the updated belief (posterior distribution) influence your decision?
What would you do if the posterior probability was only 55%?


1. Setting up the Prior Belief
To represent the initial belief of a 50% chance, I would use a Beta distribution where \bm{\alpha = \beta}. Specifically, I would choose \bm{Beta(1, 1)}.
• Why these values? In a Beta distribution, the mean is \bm{\frac{\alpha}{\alpha + \beta}}. When \bm{\alpha = 1} and \bm{\beta = 1}, the mean is exactly 0.5 (50%).
• The "Uninformative" Approach: This is known as a Flat Prior. It satisfies the 50% requirement while remaining "open-minded." It treats every possible success rate as equally likely until the actual data arrives. This ensures the final decision is driven by the experiment's results, not by a "stubborn" starting opinion.
2. Influence of the Posterior Distribution (65% Probability)
After collecting data, the analysis suggests a 65% probability that the new feature is better.
• Interpretation: While 65% is higher than the initial 50%, it represents weak evidence in a statistical sense. It indicates a positive trend, but it is not "decisive."
• Decision: I would continue the test. In professional A/B testing, we typically wait for the posterior probability to reach a threshold of 95% or higher (the "Probability of Being Best") before committing to a full rollout.
3. What if the Posterior was only 55%?
If the probability only moved from 50% to 55%:
• Interpretation: The data has barely moved the needle. This suggests the feature has a "negligible effect size"—meaning it isn't actually making a noticeable difference for users.
• Action: I would stop the experiment and "kill" the feature. Rolling out a change for a mere 5% gain in confidence isn't worth the engineering effort or the risk of complicating the app.

Exercise 6: Implementing Adaptive Experimentation
You’re running a test with three different website layouts to increase user engagement. Initially, each layout gets 33% of the traffic. After the first week, Layout C shows higher engagement.

Explain how you would adjust the traffic allocation after the first week.
Describe how you would continue to adapt the experiment in the following weeks.
What challenges might you face with adaptive experimentation, and how would you address them?

1. Adjusting Traffic Allocation
Since Layout C shows higher engagement after the first week, I would transition from a fixed split (33/33/33) to an Exploitation vs. Exploration strategy.
• The Adjustment: I would increase the traffic directed to Layout C (the leader) while reducing the traffic to Layouts A and B. For example, a new split could be 20/20/60, where 60% of users see the winning Layout C.
• The Goal: This allows the app to benefit from the higher engagement of Layout C immediately (Exploitation) while still testing A and B to see if their performance improves (Exploration).
2. Continuing the Adaptation
In the following weeks, I would use an algorithm like Thompson Sampling or Epsilon-Greedy to automate the shifts:
• Dynamic Re-weighting: If Layout C continues to win, its traffic share would climb further (e.g., to 80% or 90%).
• Safety Net: If Layout A or B suddenly starts performing better (perhaps due to a weekend trend or a specific user segment), the algorithm would automatically detect the shift and begin routing more traffic back to them.

Challenges and Solutions
1. Statistical Noise:short-term fluctuations might make layout look like a winner by pure luck.
solution: Using a minimum sample size  or a 'warm-up' period brfore the first traffic adjustment

2. Novelty Effect: users might  click layout because of curiosity but engagement might drop later.
Solution:Monitoring engagement over a long priod of time and track retention, not just initial clicks.

3. Seasonality: A layout might perform better on weekend but worse on on week days.
Solution: Ensuring the experiment runs for at least one full business cycle before making drastic changes.

4. Delayed feedback: If engagement is defined as a purchase that happens days later, the data lags.
solution:using intermediate metrics like 'Add to cart' to make faster adjustments while waiting for final data.